# BladeMask — Labelling Subspaces

**Part II · Geometric Algebra** — Tutorial 10

This tutorial introduces `BladeMask`, the ordered set of **blade IDs** that labels
matrix rows, tensor axes, and solver dimensions throughout pytanga. A blade mask
answers one question: *which basis blades does this computation care about?*

By the end you will be able to:

- Build masks from **blade IDs**, **string expressions**, **grade filters**, and
  the **full-algebra** factory.
- Extract masks directly from multivectors with `from_mv` / `from_array`.
- Test membership in O(1) (`bid in mask`) and read positions with `index()`.
- List blade names with `names()` and combine masks with `union` / `intersection`.
- Explain why masks from **different algebras** cannot be mixed.
- Derive result and unknown subspaces with `product_blade_mask` and
  `inverse_blade_mask`.

> **Prerequisites:** [Tutorial 02](../02_algebra_core/) (blade bitmasks, grades).
> Blade masks underpin [Tutorial 11](../11_equation_solving/) (the solver),
> [Tutorial 12](../12_matrix/) and [Tutorial 13](../13_tensor/).

## 1. Setup

`BladeMask` is re-exported from the top-level `pytanga` package. The prediction
helpers live in `pytanga.blade_mask.predict`. We bind one `BasisE3` algebra and
reuse it everywhere — masks are tied to a specific algebra **instance**.

In [1]:
from pytanga import BladeMask, EProduct
from pytanga.basis import BasisE2, BasisE3
from pytanga.blade_mask.predict import inverse_blade_mask, product_blade_mask

E3 = BasisE3()      # bind once; masks compare algebras by identity
E2 = BasisE2()

## 2. Blade IDs — the binary encoding

Every basis blade is identified by an integer **bitmask**: bit `k` is set when the
blade contains basis vector `e_{k+1}`. In 3D:

| Blade | Bitmask | Blade | Bitmask |
|---|---|---|---|
| `s` (scalar) | `0` | `e3` | `4` (0b100) |
| `e1` | `1` (0b001) | `e13` | `5` (0b101) |
| `e2` | `2` (0b010) | `e23` | `6` (0b110) |
| `e12` | `3` (0b011) | `I` (pseudoscalar) | `7` (0b111) |

`BasisE3` exposes these as class-level constants.

In [2]:
print("E3.E1   =", BasisE3.E1)
print("E3.E12  =", BasisE3.E12)
print("E3.E123 =", BasisE3.E123, " (pseudoscalar)")

E3.E1   = 1
E3.E12  = 3
E3.E123 = 7  (pseudoscalar)


## 3. Construction

The most general form is `BladeMask(algebra, ids=..., grades=...)`, where `ids`
and `grades` are **unioned**. Strings are parsed with the MV string parser — signs
and coefficients are discarded, only the blade names matter.

In [3]:
from_ids = BladeMask(E3, [0, 1, 2, 4])         # s, e1, e2, e3
from_str = BladeMask(E3, "1 + e12 + e23")       # scalar + e12 + e23
from_strs = BladeMask(E3, ["e12", "1 + e13"])   # union of both strings

print("from ids     :", from_ids.ids)
print("from string  :", from_str.ids)
print("from strings :", from_strs.ids)

from ids     : [0, 1, 2, 4]
from string  : [0, 3, 6]
from strings : [0, 3, 5]


In [4]:
vecs = BladeMask(E3, grades=[1])        # all vectors
bivs = BladeMask(E3, grades=[2])        # all bivectors
even = BladeMask(E3, grades=[0, 2])     # even sub-algebra
both = BladeMask(E3, "e1", grades=[2])  # e1 plus all bivectors
full = BladeMask.full(E3)               # all 2^3 = 8 blades

print("grades [1]     :", vecs.ids)
print("grades [2]     :", bivs.ids)
print("grades [0, 2]  :", even.ids)
print("e1 + grades[2] :", both.ids)
print("full           :", full.ids)

grades [1]     : [1, 2, 4]
grades [2]     : [3, 5, 6]
grades [0, 2]  : [0, 3, 5, 6]
e1 + grades[2] : [1, 3, 5, 6]
full           : [0, 1, 2, 3, 4, 5, 6, 7]


## 4. From multivectors

`BladeMask(mv)` (or `BladeMask.from_mv(mv)`) collects the **non-zero** blades of a
multivector. `BladeMask([mv1, mv2])` (or `BladeMask.from_array([...])`) unions the
non-zero blades across a list — the pattern the solver uses to size product
matrices. Structural zeros (blades present with coefficient 0) are dropped.

In [5]:
a = E3("2 e1 - 3 e12 + 0 e2")          # e2 has a structural zero

print("BladeMask(a)         :", BladeMask(a).ids)
print("BladeMask.from_mv(a) :", BladeMask.from_mv(a).ids)

b = E3("e12 + e2")
print("BladeMask.from_array([a, b]):", BladeMask.from_array([a, b]).ids)
print("BladeMask([a, b])            :", BladeMask([a, b]).ids)

BladeMask(a)         : [1, 3]
BladeMask.from_mv(a) : [1, 3]
BladeMask.from_array([a, b]): [1, 2, 3]
BladeMask([a, b])            : [1, 2, 3]


## 5. Properties — ids, size, names

- `ids` — sorted blade IDs (a copy).
- `len(mask)` — number of blades; the dimension of any axis it labels.
- `names()` — blade names sorted by grade (scalars first, then vectors, …).
- `repr(mask)` — a compact `BladeMask(['s', 'e1', …])`.
- `algebra` — the owning algebra.

In [6]:
mask = BladeMask(E3, "1 + e12 + e23")

print("ids     :", mask.ids)
print("len     :", len(mask))
print("names   :", mask.names())
print("repr    :", repr(mask))
print("algebra :", type(mask.algebra).__name__)

ids     : [0, 3, 6]
len     : 3
names   : ['s', 'e12', 'e23']
repr    : BladeMask(['s', 'e12', 'e23'])
algebra : BasisE3


## 6. Membership and position — O(1)

Membership (`bid in mask`) and position (`mask.index(bid)`) are both O(1) because
the mask keeps a companion `{blade_id: position}` dict. `index()` is the
row/column position of a blade in any matrix labelled by this mask, and raises
`KeyError` for a missing blade.

In [7]:
full = BladeMask.full(E3)

print("3 in full      :", 3 in full)           # e12 present
print("full.index(3)  :", full.index(3))        # position in sorted ids
print("full.index(7)  :", full.index(7))        # pseudoscalar is last

try:
    full.index(99)
except KeyError as e:
    print("index(99) raises:", type(e).__name__)

3 in full      : True
full.index(3)  : 3
full.index(7)  : 7
index(99) raises: KeyError


## 7. Set operations — union and intersection

Both return a new, sorted, deduplicated mask.

In [8]:
left = BladeMask(E3, [0, 1, 2])   # s, e1, e2
right = BladeMask(E3, [2, 3])     # e2, e12

print("left                     :", left.ids)
print("right                    :", right.ids)
print("left.union(right)        :", left.union(right).ids)
print("left.intersection(right) :", left.intersection(right).ids)

left                     : [0, 1, 2]
right                    : [2, 3]
left.union(right)        : [0, 1, 2, 3]
left.intersection(right) : [2]


## 8. Algebra affinity

Every mask is tied to the algebra **instance** it was built from. Operations that
combine masks (`union`, `intersection`, `product_blade_mask`,
`inverse_blade_mask`) assert that both masks share the same algebra object and
raise `AssertionError` otherwise. This prevents silently mixing blades from
different models. The check is by *identity* — bind one algebra and reuse it.

In [9]:
a_e3 = BladeMask(E3, grades=[1])
b_e3 = BladeMask(E3, grades=[2])
b_e2 = BladeMask(E2, grades=[1])    # different algebra!

print("same algebra union:", a_e3.union(b_e3).ids)

try:
    a_e3.union(b_e2)
except AssertionError as e:
    print("union across algebras raises:", e)

same algebra union: [1, 2, 3, 4, 5, 6]
union across algebras raises: Cannot union BladeMasks from different algebras


## 9. Predicting subspaces — `product_blade_mask` / `inverse_blade_mask`

These helpers perform the same bitmask algebra the solvers use internally.

- `product_blade_mask(a_mask, b_mask, product=...)` predicts which blades the
  result `A ∘ B` can occupy.
- `inverse_blade_mask(a_mask, c_mask, product=...)` predicts the maximal subspace
  the unknown `X` can inhabit in `A ∘ X = C`.

`product` is `'gp'`, `'op'`, or `'ip'` (the `EProduct` enum). `left` is currently
for API symmetry only — GP/OP/IP masks are sign-symmetric, so operand order does
not change the blade set.

In [10]:
vecs = BladeMask(E3, grades=[1])   # e1, e2, e3
e1 = BladeMask(E3, "e1")
scalar = BladeMask(E3, grades=[0])

print("EProduct values:", [e.value for e in EProduct])
print()

# A in {e1,e2,e3} times X = e1:
#   e1*e1 = 1, e2*e1 = -e12, e3*e1 = -e13
print("GP (A * e1) ->", product_blade_mask(vecs, e1, product="gp").ids)
print("OP (A ^ e1) ->", product_blade_mask(vecs, e1, product="op").ids)
print("IP (A | e1) ->", product_blade_mask(vecs, e1, product="ip").ids)

print()
# Solve A * X = scalar 1  →  X must be a vector (same subspace as A).
print("inverse (A * X = s) ->", inverse_blade_mask(vecs, scalar, product="gp").ids)

EProduct values: ['gp', 'ip', 'op']

GP (A * e1) -> [0, 3, 5]
OP (A ^ e1) -> [3, 5]
IP (A | e1) -> [0]

inverse (A * X = s) -> [1, 2, 4]


In [11]:
# `complete=True` grows the result mask to its fixed point. Multiplying the
# vector set {e1,e2,e3} by the bivector e12 reaches e1, e2 (in-plane) and
# e123 (e12 * e3), so the closure is {e1, e2, e3, e123}.
e12 = BladeMask(E3, "e12")
print("GP closure of vectors under e12:",
      product_blade_mask(e12, vecs, product="gp", complete=True).ids)

GP closure of vectors under e12: [1, 2, 4, 7]


## 10. Summary & next steps

| Concept | API |
|---|---|
| From blade IDs | `BladeMask(E3, [0, 1, 3])` |
| From string(s) | `BladeMask(E3, "1 + e12")`, `BladeMask(E3, ["e1", "e12"])` |
| By grade | `BladeMask(E3, grades=[1])` |
| Full algebra | `BladeMask.full(E3)` |
| From MV / list of MVs | `BladeMask(mv)`, `BladeMask([a, b])` |
| Membership / position | `bid in mask`, `mask.index(bid)` |
| Names / size | `mask.names()`, `len(mask)` |
| Set ops | `mask.union(other)`, `mask.intersection(other)` |
| Predict result / unknown | `product_blade_mask(a, b)`, `inverse_blade_mask(a, c)` |

**Where to go next:**

- [**11 · Equation Solving**](../11_equation_solving/) — how the solver uses these
  masks to build and solve linear systems.
- [**12 · Matrix Operations**](../12_matrix/) and
  [**13 · Tensor Operations**](../13_tensor/) — where blade masks label matrix
  rows and tensor axes.